# JobHunter AI - Multi-Agent Job Application System

This notebook runs the full job hunting pipeline:
1. **Phase 1**: Crawl job boards (Agents 1-5)
2. **Phase 2**: Extract job requirements (Agent 6)
3. **Phase 3**: Generate tailored CVs and cover letters (Agent 7)
4. **Phase 4**: Evaluate job fit (Agent 8)

## Cell 1: Setup and Configuration
Load config, connect to local Excel storage, and initialise AI.

In [ ]:
import sys
import os

from sheets_manager import load_config, SheetsManager
from ai_helper import AIHelper

config = load_config("config/config.json")

file_path = config.get("storage", {}).get("file_path", "data/jobs.xlsx")
sheets = SheetsManager(file_path=file_path)
sheets.connect()

ai = AIHelper(
    api_key=config["ai_provider"]["api_key"],
    base_url=config["ai_provider"]["base_url"],
    model=config["ai_provider"]["model"]
)

print("Setup complete.")

## Phase 1: Job Discovery (Agents 1-5)
Crawl multiple job boards in parallel.

In [ ]:
from crawlers.runner import run_all_crawlers, run_selected_crawlers

JOB_TITLE = "Data Analyst"
LOCATION = "United Kingdom"

filters = {
    "job_type": config["search_defaults"]["job_type"],
    "date_posted": config["search_defaults"]["date_posted"]
}

jobs = run_all_crawlers(config, JOB_TITLE, LOCATION, filters)

print(f"\nTotal jobs found: {len(jobs)}")

In [ ]:
import pandas as pd

if jobs:
    new_jobs = []
    for job in jobs:
        if not sheets.is_duplicate(job["Job URL"]):
            new_jobs.append(job)
    
    if new_jobs:
        sheets.add_jobs_batch(new_jobs)
        print(f"Added {len(new_jobs)} new jobs to Excel file.")
    else:
        print("All jobs already exist in the file.")
    
    df = pd.DataFrame(jobs)
    display(df)
else:
    print("No jobs found.")

## Phase 2: Job Requirements Extraction (Agent 6)
Visit each job URL and extract structured requirements using AI.

In [ ]:
from agents.requirements_extractor import RequirementsExtractor

extractor = RequirementsExtractor(ai, config)
processed_count = extractor.process_jobs_from_sheet(sheets)

print(f"\nProcessed {processed_count} jobs.")

## Phase 3: CV and Cover Letter Generation (Agent 7)
Generate tailored CVs and cover letters in LaTeX, compiled to PDF.

In [ ]:
from agents.cv_generator import CVGenerator

generator = CVGenerator(ai, config)
generated_count = generator.process_jobs_from_sheet(sheets)

print(f"\nGenerated documents for {generated_count} jobs.")

## Phase 4: Fit Evaluation (Agent 8)
Score how well your profile matches each job.

In [ ]:
from agents.fit_evaluator import FitEvaluator

evaluator = FitEvaluator(ai, config)
evaluated_count = evaluator.process_jobs_from_sheet(sheets)

print(f"\nEvaluated {evaluated_count} jobs.")

## View Results
Display all jobs with their fit scores.

In [ ]:
df = sheets.get_all_jobs()

if not df.empty:
    df["Fit Score"] = pd.to_numeric(df["Fit Score"], errors="coerce").fillna(0)
    df_sorted = df.sort_values("Fit Score", ascending=False)
    display(df_sorted[["Job Title", "Company", "Location", "Source", "Fit Score", "Status"]].head(20))
else:
    print("No data yet.")